In [1]:
import os
import requests
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import gzip
import time

print("=" * 60)
print("RNA-SEQ DATA DOWNLOAD - MMRF CoMMpass")
print("FULL DATASET: All 787 Samples")
print("=" * 60)

# ============================================================
# STEP 1: Setup Directories
# ============================================================
print("\n1️⃣ Setting up directories...")

PROJECT_ROOT = Path.cwd().parent
RNASEQ_DIR = PROJECT_ROOT / 'data' / 'raw' / 'rnaseq'
COUNTS_DIR = RNASEQ_DIR / 'counts'

# Create directories
RNASEQ_DIR.mkdir(parents=True, exist_ok=True)
COUNTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ RNA-seq directory: {RNASEQ_DIR}")
print(f"✅ Counts directory: {COUNTS_DIR}")

# ============================================================
# STEP 2: Query GDC for RNA-seq Files (DIAGNOSTIC VERSION)
# ============================================================
print("\n2️⃣ Querying GDC for RNA-seq files...")

GDC_API = "https://api.gdc.cancer.gov"
FILES_ENDPOINT = f"{GDC_API}/files"

# First, let's check what's available for MMRF-COMMPASS
print("\n🔍 Checking available data types for MMRF-COMMPASS...")

# Simple query first
simple_filters = {
    "op": "in",
    "content": {
        "field": "cases.project.project_id",
        "value": ["MMRF-COMMPASS"]
    }
}

params = {
    "filters": json.dumps(simple_filters),
    "facets": "data_category,data_type,analysis.workflow_type",
    "size": 0
}

response = requests.get(FILES_ENDPOINT, params=params)
print(f"Response status: {response.status_code}")

if response.status_code == 200:
    facets = response.json()['data']['aggregations']
    
    print("\n📊 Available Data Categories:")
    for item in facets.get('data_category', {}).get('buckets', []):
        print(f"   - {item['key']}: {item['doc_count']} files")
    
    print("\n📊 Available Data Types:")
    for item in facets.get('data_type', {}).get('buckets', []):
        print(f"   - {item['key']}: {item['doc_count']} files")
    
    print("\n📊 Available Workflow Types:")
    for item in facets.get('analysis.workflow_type', {}).get('buckets', []):
        print(f"   - {item['key']}: {item['doc_count']} files")
else:
    print(f"❌ API Error: {response.status_code}")
    print(response.text)

# Now try the actual query with correct filters
print("\n🔍 Querying for RNA-seq count files...")

filters = {
    "op": "and",
    "content": [
        {
            "op": "in",
            "content": {
                "field": "cases.project.project_id",
                "value": ["MMRF-COMMPASS"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "data_category",
                "value": ["Transcriptome Profiling"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "data_type",
                "value": ["Gene Expression Quantification"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "analysis.workflow_type",
                "value": ["STAR - Counts"]  # Try STAR instead of HTSeq
            }
        }
    ]
}

fields = [
    "file_id",
    "file_name",
    "file_size",
    "cases.case_id",
    "cases.submitter_id"
]

params = {
    "filters": json.dumps(filters),
    "fields": ",".join(fields),
    "format": "JSON",
    "size": 1000
}

response = requests.get(FILES_ENDPOINT, params=params)

if response.status_code != 200:
    print(f"❌ API Error: {response.status_code}")
    print(response.text[:500])
    raise Exception(f"API request failed: {response.status_code}")

result = response.json()
data = result.get('data', {}).get('hits', [])

print(f"✅ Found {len(data)} RNA-seq files with STAR - Counts")

# If STAR didn't work, try HTSeq
if len(data) == 0:
    print("\n🔍 Trying HTSeq - Counts instead...")
    
    filters['content'][3]['content']['value'] = ["HTSeq - Counts"]
    params['filters'] = json.dumps(filters)
    
    response = requests.get(FILES_ENDPOINT, params=params)
    result = response.json()
    data = result.get('data', {}).get('hits', [])
    
    print(f"✅ Found {len(data)} RNA-seq files with HTSeq - Counts")

# If still nothing, try without workflow filter
if len(data) == 0:
    print("\n🔍 Trying without workflow type filter...")
    
    filters = {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "cases.project.project_id", "value": ["MMRF-COMMPASS"]}},
            {"op": "in", "content": {"field": "data_category", "value": ["Transcriptome Profiling"]}},
            {"op": "in", "content": {"field": "data_type", "value": ["Gene Expression Quantification"]}}
        ]
    }
    
    params['filters'] = json.dumps(filters)
    response = requests.get(FILES_ENDPOINT, params=params)
    result = response.json()
    data = result.get('data', {}).get('hits', [])
    
    print(f"✅ Found {len(data)} RNA-seq files (any workflow)")

if len(data) == 0:
    print("\n❌ ERROR: No RNA-seq files found!")
    print("This might be a network issue or the data structure has changed.")
    print("Please share this output so we can debug further.")
    raise Exception("No RNA-seq files found")
# ============================================================
# STEP 3: Create File Manifest
# ============================================================
print("\n3️⃣ Creating download manifest...")

# Parse file information
manifest = []
for item in data:
    manifest.append({
        'file_id': item['file_id'],
        'file_name': item['file_name'],
        'file_size_mb': item['file_size'] / 1e6,
        'case_id': item['cases'][0]['case_id'],
        'submitter_id': item['cases'][0]['submitter_id']
    })

manifest_df = pd.DataFrame(manifest)

# Save manifest
manifest_path = RNASEQ_DIR / 'rnaseq_file_manifest.csv'
manifest_df.to_csv(manifest_path, index=False)

print(f"✅ Manifest saved: {manifest_path}")
print(f"\n📊 Dataset Statistics:")
print(f"   Total files: {len(manifest_df)}")
print(f"   Total size: {manifest_df['file_size_mb'].sum():.1f} MB (~{manifest_df['file_size_mb'].sum()/1024:.2f} GB)")
print(f"   Unique patients: {manifest_df['case_id'].nunique()}")
print(f"   Average file size: {manifest_df['file_size_mb'].mean():.2f} MB")

# Display sample
print("\n📋 Sample of files to download:")
print(manifest_df.head())

# ============================================================
# STEP 4: Download ALL Files
# ============================================================
print("\n" + "=" * 60)
print("4️⃣ DOWNLOADING ALL RNA-SEQ FILES")
print("=" * 60)

print(f"\n📥 Preparing to download {len(manifest_df)} files")
print(f"   Total size: {manifest_df['file_size_mb'].sum():.1f} MB (~{manifest_df['file_size_mb'].sum()/1024:.2f} GB)")
print(f"   Estimated time: ~30-60 minutes")
print("\n⚠️  This will take some time. Go grab a coffee! ☕")
print("   Progress will be shown below...\n")

# Download function with retry logic
def download_file(file_id, file_name, output_dir, max_retries=3):
    """Download a single file from GDC with retry logic"""
    url = f"{GDC_API}/data/{file_id}"
    output_path = output_dir / file_name
    
    # Skip if already exists and is not empty
    if output_path.exists() and output_path.stat().st_size > 0:
        return output_path, "exists"
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, stream=True, timeout=60)
            response.raise_for_status()
            
            total_size = int(response.headers.get('content-length', 0))
            
            with open(output_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            
            # Verify file was written
            if output_path.stat().st_size > 0:
                return output_path, "success"
            else:
                if attempt < max_retries - 1:
                    time.sleep(2)
                    continue
                return None, "error: empty file"
                
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
                continue
            return None, f"error: {str(e)}"
    
    return None, "error: max retries exceeded"

# Download all files with progress bar
successful = 0
failed = 0
skipped = 0
failed_files = []

start_time = time.time()

# Use tqdm for nice progress bar
for idx, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), 
                     desc="Downloading RNA-seq files", unit="file"):
    
    path, status = download_file(row['file_id'], row['file_name'], COUNTS_DIR)
    
    if status == "success":
        successful += 1
    elif status == "exists":
        skipped += 1
    else:
        failed += 1
        failed_files.append({
            'file_name': row['file_name'],
            'file_id': row['file_id'],
            'status': status
        })
    
    # Print progress every 50 files
    if (successful + skipped + failed) % 50 == 0:
        elapsed = time.time() - start_time
        rate = (successful + skipped) / elapsed if elapsed > 0 else 0
        remaining = len(manifest_df) - (successful + skipped + failed)
        eta = remaining / rate if rate > 0 else 0
        print(f"\n   Progress: {successful + skipped}/{len(manifest_df)} | "
              f"ETA: {eta/60:.1f} min | "
              f"Rate: {rate:.1f} files/sec")

elapsed_time = time.time() - start_time

print("\n" + "=" * 60)
print("DOWNLOAD COMPLETE!")
print("=" * 60)
print(f"✅ Successful downloads: {successful}")
print(f"⏭️  Skipped (already existed): {skipped}")
print(f"❌ Failed: {failed}")
print(f"⏱️  Total time: {elapsed_time/60:.1f} minutes")
print(f"📁 Files saved to: {COUNTS_DIR}")

# Save failed files list if any
if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_path = RNASEQ_DIR / 'failed_downloads.csv'
    failed_df.to_csv(failed_path, index=False)
    print(f"\n⚠️  Failed files saved to: {failed_path}")
    print("   You can retry these later if needed")

# ============================================================
# STEP 5: Verify Downloads
# ============================================================
print("\n5️⃣ Verifying downloads...")

# Check downloaded files
downloaded_files = list(COUNTS_DIR.glob('*.gz'))
print(f"✅ Found {len(downloaded_files)} files on disk")

# Calculate total size
total_size = sum(f.stat().st_size for f in downloaded_files) / 1e9
print(f"📊 Total data size: {total_size:.2f} GB")

# Verify file integrity - check if files are not empty
empty_files = [f for f in downloaded_files if f.stat().st_size < 1000]
if empty_files:
    print(f"⚠️  Warning: {len(empty_files)} files appear empty or corrupted")
else:
    print(f"✅ All files appear valid (>1KB)")

# Show sample of a file
if downloaded_files:
    sample_file = downloaded_files[0]
    print(f"\n📄 Sample file inspection:")
    print(f"   File: {sample_file.name}")
    print(f"   Size: {sample_file.stat().st_size / 1e6:.2f} MB")
    
    # Read first 10 lines
    print(f"\n   First 10 lines (gene counts):")
    with gzip.open(sample_file, 'rt') as f:
        for i, line in enumerate(f):
            if i >= 10:
                break
            parts = line.strip().split('\t')
            if len(parts) == 2:
                print(f"      {parts[0]}: {parts[1]}")

# ============================================================
# STEP 6: Match with Clinical Data
# ============================================================
print("\n6️⃣ Matching RNA-seq with clinical data...")

# Load clinical data
clinical_path = PROJECT_ROOT / 'data' / 'processed' / 'patient_level_clean.csv'
if clinical_path.exists():
    clinical_df = pd.read_csv(clinical_path)
    
    # Match case IDs
    clinical_cases = set(clinical_df['cases.case_id'])
    rnaseq_cases = set(manifest_df['case_id'])
    
    overlap = clinical_cases & rnaseq_cases
    
    print(f"📊 Data Integration:")
    print(f"   Patients with clinical data: {len(clinical_cases)}")
    print(f"   Patients with RNA-seq data: {len(rnaseq_cases)}")
    print(f"   Overlap (have both): {len(overlap)}")
    print(f"   Match rate: {len(overlap)/len(clinical_cases)*100:.1f}%")
    
    # Save integrated patient list
    integrated_patients = manifest_df[manifest_df['case_id'].isin(overlap)].copy()
    integrated_path = RNASEQ_DIR / 'patients_with_clinical_and_rnaseq.csv'
    integrated_patients.to_csv(integrated_path, index=False)
    print(f"✅ Integrated patient list saved: {integrated_path}")
else:
    print("⚠️  Clinical data not found - will integrate later")

print("\n" + "=" * 60)
print("✅ RNA-SEQ DATA DOWNLOAD COMPLETE!")
print("=" * 60)
print("\n📊 Final Summary:")
print(f"   ✅ Files downloaded: {successful + skipped}")
print(f"   📁 Location: {COUNTS_DIR}")
print(f"   📋 Manifest: {manifest_path}")
print(f"   💾 Total size: {total_size:.2f} GB")
print(f"   👥 Patients: {manifest_df['case_id'].nunique()}")
print("\n🚀 Next Steps:")
print("   1. Create notebook 03: Process and merge count files into matrix")
print("   2. Quality control and normalization")
print("   3. Differential expression analysis")
print("   4. Survival-gene correlation analysis")
print("   5. Machine learning biomarker discovery")
print("\n💡 Tip: This data is now ready for comprehensive genomic analysis!")


RNA-SEQ DATA DOWNLOAD - MMRF CoMMpass
FULL DATASET: All 787 Samples

1️⃣ Setting up directories...
✅ RNA-seq directory: C:\Users\Abhinav\Mylo\myeloma-biomarker-project\data\raw\rnaseq
✅ Counts directory: C:\Users\Abhinav\Mylo\myeloma-biomarker-project\data\raw\rnaseq\counts

2️⃣ Querying GDC for RNA-seq files...

🔍 Checking available data types for MMRF-COMMPASS...
Response status: 200

📊 Available Data Categories:
   - simple nucleotide variation: 18304 files
   - sequencing reads: 6577 files
   - structural variation: 3458 files
   - somatic structural variation: 2032 files
   - copy number variation: 2020 files
   - transcriptome profiling: 1718 files

📊 Available Data Types:
   - Annotated Somatic Mutation: 9746 files
   - Aligned Reads: 6577 files
   - Raw Simple Somatic Mutation: 6376 files
   - Transcript Fusion: 3436 files
   - Structural Rearrangement: 2054 files
   - Aggregated Somatic Mutation: 1091 files
   - Masked Somatic Mutation: 1091 files
   - Copy Number Segment: 101


   Progress: 50/859 | ETA: 145.9 min | Rate: 0.1 files/sec



   Progress: 100/859 | ETA: 129.0 min | Rate: 0.1 files/sec



   Progress: 150/859 | ETA: 96.1 min | Rate: 0.1 files/sec



   Progress: 200/859 | ETA: 77.2 min | Rate: 0.1 files/sec


...


   Progress: 700/859 | ETA: 12.2 min | Rate: 0.2 files/sec



   Progress: 750/859 | ETA: 8.3 min | Rate: 0.2 files/sec



   Progress: 800/859 | ETA: 4.4 min | Rate: 0.2 files/sec



   Progress: 850/859 | ETA: 0.7 min | Rate: 0.2 files/sec



DOWNLOAD COMPLETE!
✅ Successful downloads: 859
⏭️  Skipped (already existed): 0
❌ Failed: 0
⏱️  Total time: 63.3 minutes
📁 Files saved to: C:\Users\Abhinav\Mylo\myeloma-biomarker-project\data\raw\rnaseq\counts

5️⃣ Verifying downloads...
✅ Found 0 files on disk
📊 Total data size: 0.00 GB
✅ All files appear valid (>1KB)

6️⃣ Matching RNA-seq with clinical data...
📊 Data Integration:
   Patients with clinical data: 995
   Patients with RNA-seq data: 787
   Overlap (have both): 787
   Match rate: 79.1%
✅ Integrated patient list saved: C:\Users\Abhinav\Mylo\myeloma-biomarker-project\data\raw\rnaseq\patients_with_clinical_and_rnaseq.csv

✅ RNA-SEQ DATA DOWNLOAD COMPLETE!

📊 Final Summary:
   ✅ Files downloaded: 859
   📁 Location: C:\Users\Abhinav\Mylo\myeloma-biomarker-project\data\raw\rnaseq\counts
   📋 Manifest: C:\Users\Abhinav\Mylo\myeloma-biomarker-project\data\raw\rnaseq\rnaseq_file_manifest.csv
   💾 Total size: 0.00 GB
   👥 Patients: 787

🚀 Next Steps:
   1. Create notebook 03: Proc

In [3]:
import os
from pathlib import Path

# Check the directory
COUNTS_DIR = Path(r'C:\Users\Abhinav\Mylo\myeloma-biomarker-project\data\raw\rnaseq\counts')

print("🔍 Investigating download directory...")
print(f"Directory exists: {COUNTS_DIR.exists()}")
print(f"Directory path: {COUNTS_DIR}")

if COUNTS_DIR.exists():
    # List all files
    all_files = list(COUNTS_DIR.glob('*'))
    print(f"\n📁 Files in directory: {len(all_files)}")
    
    if all_files:
        print(f"\n📋 First 10 files:")
        for f in all_files[:10]:
            print(f"   {f.name} - {f.stat().st_size / 1e6:.2f} MB")
    else:
        print("❌ Directory is empty!")
        
    # Check parent directory
    print(f"\n📁 Parent directory contents:")
    parent_files = list(COUNTS_DIR.parent.glob('*'))
    for f in parent_files:
        if f.is_file():
            print(f"   FILE: {f.name}")
        else:
            print(f"   DIR:  {f.name}/")
else:
    print("❌ Directory doesn't exist!")

# Check if files were downloaded to wrong location
print("\n🔍 Checking for .gz files in project...")
project_root = Path(r'C:\Users\Abhinav\Mylo\myeloma-biomarker-project')
gz_files = list(project_root.rglob('*.gz'))
print(f"Found {len(gz_files)} .gz files in project")

if gz_files:
    print("\n📍 Files found at:")
    for f in gz_files[:5]:
        print(f"   {f}")

🔍 Investigating download directory...
Directory exists: True
Directory path: C:\Users\Abhinav\Mylo\myeloma-biomarker-project\data\raw\rnaseq\counts

📁 Files in directory: 859

📋 First 10 files:
   000c5024-cee6-4ff6-bb79-312ccaf49572.rna_seq.augmented_star_gene_counts.tsv - 4.23 MB
   006d7516-68f0-4126-8511-31e71c7710b6.rna_seq.augmented_star_gene_counts.tsv - 4.23 MB
   00809c1b-3729-4aa9-8d9b-fb459020651d.rna_seq.augmented_star_gene_counts.tsv - 4.21 MB
   010c2e72-a53d-4daa-8d8e-4db09ef3a94a.rna_seq.augmented_star_gene_counts.tsv - 4.23 MB
   0122dd67-7ded-4cff-9133-34890b477748.rna_seq.augmented_star_gene_counts.tsv - 4.22 MB
   0126848c-f9d5-4d37-a8ee-c96ffd309a6a.rna_seq.augmented_star_gene_counts.tsv - 4.21 MB
   01c58776-fced-4938-963b-2f0741aae5da.rna_seq.augmented_star_gene_counts.tsv - 4.21 MB
   01f36fd2-bb3b-4f13-8437-74f0321e25a4.rna_seq.augmented_star_gene_counts.tsv - 4.22 MB
   01fa76c9-5320-44b1-973f-f7cd730b3a7a.rna_seq.augmented_star_gene_counts.tsv - 4.23 MB
   02